# 01b - Automatic Player Collection

The first data collection notebook used a small manually selected player list.

That sample was useful for testing the pipeline, but it is too small and biased for meaningful clustering.

This notebook automatically collects a larger and more diverse set of Old School RuneScape player names using random partial username searches.

After collecting player names, the notebook downloads public OSRS Hiscores data for each player and saves the result as a structured dataset.

## 1. Import required packages

The following packages are used for:

- API requests,
- data handling,
- random search term generation,
- waiting between requests,
- file path management.

In [35]:
import requests
import pandas as pd
import time
import random
import string

from pathlib import Path

## 2. Project paths

The collected dataset will be saved into the `data/processed` folder.

The folder structure follows the project layout used throughout the repository.

In [36]:
## 2. Project paths

The collected dataset will be saved into the `data/processed` folder.

The folder structure follows the project layout used throughout the repository.

SyntaxError: invalid syntax (3039097701.py, line 3)

## 3. Why pseudo-random player collection?

Using only top leaderboard players would introduce strong sampling bias.

Top-ranked players usually have very high levels and similar progression patterns, which would make clustering less meaningful.

Instead, this notebook uses random partial username searches to collect a more diverse set of players.

This is not a perfectly random sample of all OSRS players, but it is more suitable for this project than a top leaderboard sample.

## 4. Generate random username search terms

The player collection process starts by generating random short character combinations.

These terms will be used as partial username searches.

In [37]:
def generate_random_search_terms(
    n_terms=300,
    min_length=2,
    max_length=4,
    random_state=42
):
    """
    Generate random partial username search terms.

    Examples:
    - ab
    - x7
    - kq2
    - 9rs

    These terms are later used to search for OSRS player names.
    """
    random.seed(random_state)

    terms = set()
    characters = string.ascii_lowercase + string.digits

    while len(terms) < n_terms:
        length = random.randint(min_length, max_length)
        term = "".join(random.choice(characters) for _ in range(length))
        terms.add(term)

    return list(terms)

In [38]:
search_terms = generate_random_search_terms(
    n_terms=20,
    min_length=2,
    max_length=4,
    random_state=42
)

search_terms[:10]

['yr9', 'ig', 'yg', '6b', 'fn', '38hy', 'xmec', 'k8pk', 'nre', 'rak']

## 5. Search players by partial username

This function sends a request to the Wise Old Man player search endpoint.

For each random search term, it returns matching player names.

In [39]:
def search_wom_players(username_query, limit=20):
    """
    Search Wise Old Man players by partial username.

    Parameters:
    - username_query: partial username search term
    - limit: maximum number of returned players

    Returns:
    - list of player names
    """
    url = "https://api.wiseoldman.net/v2/players/search"

    params = {
        "username": username_query,
        "limit": limit
    }

    try:
        response = requests.get(url, params=params, timeout=15)
        response.raise_for_status()

        data = response.json()

        player_names = []

        for item in data:
            username = (
                item.get("displayName")
                or item.get("username")
            )

            if username:
                player_names.append(username)

        return player_names

    except requests.RequestException as error:
        print(f"Search failed for query '{username_query}': {error}")
        return []

In [40]:
test_names = search_wom_players("rs", limit=10)

len(test_names), test_names[:10]

(10,
 ['RsSuomi',
  'Rs Jamal',
  'Rs But Afk',
  'rsvp',
  'Rs Wiki',
  'RS SLAG',
  'Rs Kajcsa',
  'rsyux',
  'RS ape',
  'rs basherr'])

## 6. Collect pseudo-random player names

This function repeatedly generates search terms, queries matching players, and stores unique usernames.

The process stops when the target number of unique players is reached or when all search terms have been used.

In [41]:
def collect_pseudo_random_player_names(
    target_count=500,
    n_search_terms=500,
    results_per_term=20,
    sleep_seconds=0.2,
    random_state=42
):
    """
    Collect a diverse set of OSRS player names using random partial username searches.

    Parameters:
    - target_count: desired number of unique player names
    - n_search_terms: number of generated random search terms
    - results_per_term: maximum number of players returned per search term
    - sleep_seconds: waiting time between API requests
    - random_state: seed for reproducibility

    Returns:
    - list of unique player names
    """
    search_terms = generate_random_search_terms(
        n_terms=n_search_terms,
        min_length=2,
        max_length=4,
        random_state=random_state
    )

    collected_names = []

    for index, term in enumerate(search_terms, start=1):
        print(f"[{index}/{len(search_terms)}] Searching term: {term}")

        names = search_wom_players(
            username_query=term,
            limit=results_per_term
        )

        collected_names.extend(names)
        collected_names = list(dict.fromkeys(collected_names))

        print(f"Collected unique players: {len(collected_names)}")

        if len(collected_names) >= target_count:
            break

        time.sleep(sleep_seconds)

    return collected_names[:target_count]

## 7. Run player name collection

In this step, we collect the player names that will be used for Hiscores data collection.

The target sample size can be adjusted depending on runtime.

In [43]:
player_names = collect_pseudo_random_player_names(
    target_count=300,
    n_search_terms=600,
    results_per_term=20,
    sleep_seconds=0.2,
    random_state=42
)

len(player_names), player_names[:20]

[1/600] Searching term: 07
Collected unique players: 20
[2/600] Searching term: q7
Collected unique players: 30
[3/600] Searching term: mdry
Collected unique players: 30
[4/600] Searching term: ka
Collected unique players: 50
[5/600] Searching term: yr0
Collected unique players: 51
[6/600] Searching term: t0hi
Collected unique players: 51
[7/600] Searching term: oe3
Collected unique players: 51
[8/600] Searching term: lie3
Collected unique players: 51
[9/600] Searching term: bwe
Collected unique players: 71
[10/600] Searching term: xxyc
Collected unique players: 71
[11/600] Searching term: n5u4
Collected unique players: 71
[12/600] Searching term: zrad
Collected unique players: 74
[13/600] Searching term: t6z
Collected unique players: 74
[14/600] Searching term: ump
Collected unique players: 94
[15/600] Searching term: qc
Collected unique players: 114
[16/600] Searching term: 830
Collected unique players: 117
[17/600] Searching term: 3w
Collected unique players: 137
[18/600] Searching 

[56/600] Searching term: eiu
Search failed for query 'eiu': 429 Client Error: Too Many Requests for url: https://api.wiseoldman.net/v2/players/search?username=eiu&limit=20
Collected unique players: 157
[57/600] Searching term: s5
Search failed for query 's5': 429 Client Error: Too Many Requests for url: https://api.wiseoldman.net/v2/players/search?username=s5&limit=20
Collected unique players: 157
[58/600] Searching term: 72r
Search failed for query '72r': 429 Client Error: Too Many Requests for url: https://api.wiseoldman.net/v2/players/search?username=72r&limit=20
Collected unique players: 157
[59/600] Searching term: gup3
Search failed for query 'gup3': 429 Client Error: Too Many Requests for url: https://api.wiseoldman.net/v2/players/search?username=gup3&limit=20
Collected unique players: 157
[60/600] Searching term: hzb
Search failed for query 'hzb': 429 Client Error: Too Many Requests for url: https://api.wiseoldman.net/v2/players/search?username=hzb&limit=20
Collected unique pla

[97/600] Searching term: 4kfs
Search failed for query '4kfs': 429 Client Error: Too Many Requests for url: https://api.wiseoldman.net/v2/players/search?username=4kfs&limit=20
Collected unique players: 157
[98/600] Searching term: 852f
Search failed for query '852f': 429 Client Error: Too Many Requests for url: https://api.wiseoldman.net/v2/players/search?username=852f&limit=20
Collected unique players: 157
[99/600] Searching term: 9yv
Search failed for query '9yv': 429 Client Error: Too Many Requests for url: https://api.wiseoldman.net/v2/players/search?username=9yv&limit=20
Collected unique players: 157
[100/600] Searching term: 1vrj
Search failed for query '1vrj': 429 Client Error: Too Many Requests for url: https://api.wiseoldman.net/v2/players/search?username=1vrj&limit=20
Collected unique players: 157
[101/600] Searching term: ren
Search failed for query 'ren': 429 Client Error: Too Many Requests for url: https://api.wiseoldman.net/v2/players/search?username=ren&limit=20
Collected

[138/600] Searching term: ifet
Search failed for query 'ifet': 429 Client Error: Too Many Requests for url: https://api.wiseoldman.net/v2/players/search?username=ifet&limit=20
Collected unique players: 157
[139/600] Searching term: h3
Search failed for query 'h3': 429 Client Error: Too Many Requests for url: https://api.wiseoldman.net/v2/players/search?username=h3&limit=20
Collected unique players: 157
[140/600] Searching term: g50r
Search failed for query 'g50r': 429 Client Error: Too Many Requests for url: https://api.wiseoldman.net/v2/players/search?username=g50r&limit=20
Collected unique players: 157
[141/600] Searching term: cem9
Search failed for query 'cem9': 429 Client Error: Too Many Requests for url: https://api.wiseoldman.net/v2/players/search?username=cem9&limit=20
Collected unique players: 157
[142/600] Searching term: 73
Search failed for query '73': 429 Client Error: Too Many Requests for url: https://api.wiseoldman.net/v2/players/search?username=73&limit=20
Collected un

[179/600] Searching term: 1a78
Search failed for query '1a78': 429 Client Error: Too Many Requests for url: https://api.wiseoldman.net/v2/players/search?username=1a78&limit=20
Collected unique players: 157
[180/600] Searching term: 1z
Search failed for query '1z': 429 Client Error: Too Many Requests for url: https://api.wiseoldman.net/v2/players/search?username=1z&limit=20
Collected unique players: 157
[181/600] Searching term: bj
Search failed for query 'bj': 429 Client Error: Too Many Requests for url: https://api.wiseoldman.net/v2/players/search?username=bj&limit=20
Collected unique players: 157
[182/600] Searching term: ds
Search failed for query 'ds': 429 Client Error: Too Many Requests for url: https://api.wiseoldman.net/v2/players/search?username=ds&limit=20
Collected unique players: 157
[183/600] Searching term: tb
Search failed for query 'tb': 429 Client Error: Too Many Requests for url: https://api.wiseoldman.net/v2/players/search?username=tb&limit=20
Collected unique players

(300,
 ['07 iron',
  '07dive',
  '07 Flare',
  '07 Nihilist',
  '07 Hollyman',
  '07Mana',
  '07Ashley',
  '07wan',
  '07Chris',
  '07 0',
  '07 Preacher',
  '07 tim',
  '07 PRISMA',
  '07 t bone',
  '07Geno',
  '07TroyScape',
  '07 maisy',
  '07 ezra',
  '07 Account',
  '07 Camry'])

## 8. Save collected player names

The collected usernames are saved separately.

This makes the data collection process more reproducible and allows reusing the same player list later.

In [44]:
player_names_path = PROCESSED_DATA_DIR / "osrs_collected_player_names.csv"

pd.DataFrame({"player": player_names}).to_csv(
    player_names_path,
    index=False,
    encoding="utf-8"
)

player_names_path

WindowsPath('C:/Projects/osrs-player-segmentation/data/processed/osrs_collected_player_names.csv')

## 9. Define OSRS Hiscores skill structure

The OSRS Hiscores endpoint returns skill data in a fixed order.

Each skill row contains:

- rank,
- level,
- experience.

The skill list must match the order returned by the Hiscores endpoint.

In [45]:
SKILLS = [
    "overall",
    "attack",
    "defence",
    "strength",
    "hitpoints",
    "ranged",
    "prayer",
    "magic",
    "cooking",
    "woodcutting",
    "fletching",
    "fishing",
    "firemaking",
    "crafting",
    "smithing",
    "mining",
    "herblore",
    "agility",
    "thieving",
    "slayer",
    "farming",
    "runecraft",
    "hunter",
    "construction",
    "sailing"
]

## 10. Fetch OSRS Hiscores data

After collecting player names, this section downloads public OSRS Hiscores data for each player.

In [46]:
def fetch_hiscores(player_name):
    """
    Fetch raw OSRS Hiscores data for a single player.

    Returns:
    - raw text response if successful
    - None if the request fails
    """
    url = "https://secure.runescape.com/m=hiscore_oldschool/index_lite.ws"

    params = {
        "player": player_name
    }

    try:
        response = requests.get(url, params=params, timeout=10)

        if response.status_code == 200:
            return response.text

        print(f"Failed: {player_name} - HTTP {response.status_code}")
        return None

    except requests.RequestException as error:
        print(f"Request error for {player_name}: {error}")
        return None

## 11. Parse Hiscores skill data

The raw Hiscores response is text-based.

This function converts the skill section into a structured Python dictionary.

In [47]:
def parse_skill_data(player_name, raw_text):
    """
    Parse the skill section of OSRS Hiscores data.

    The first rows of the response contain skill data.
    Each row contains:
    rank, level, experience.
    """
    lines = raw_text.strip().splitlines()
    skill_lines = lines[:len(SKILLS)]

    player_data = {
        "player": player_name
    }

    for skill_name, line in zip(SKILLS, skill_lines):
        values = line.split(",")

        if len(values) < 3:
            continue

        rank, level, xp = values[:3]

        player_data[f"{skill_name}_rank"] = int(rank)
        player_data[f"{skill_name}_level"] = int(level)
        player_data[f"{skill_name}_xp"] = int(xp)

    return player_data

## 12. Download Hiscores data for collected players

This step iterates through the collected player list and downloads Hiscores data for each player.

Invalid or unavailable player profiles are skipped.

In [48]:
records = []
failed_players = []

for index, player_name in enumerate(player_names, start=1):
    print(f"[{index}/{len(player_names)}] Fetching Hiscores: {player_name}")

    raw_text = fetch_hiscores(player_name)

    if raw_text is None:
        failed_players.append(player_name)
        continue

    parsed_data = parse_skill_data(player_name, raw_text)
    records.append(parsed_data)

    time.sleep(0.5)

df_hiscores = pd.DataFrame(records)

df_hiscores.head()

[1/300] Fetching Hiscores: 07 iron
[2/300] Fetching Hiscores: 07dive
[3/300] Fetching Hiscores: 07 Flare
[4/300] Fetching Hiscores: 07 Nihilist
[5/300] Fetching Hiscores: 07 Hollyman
[6/300] Fetching Hiscores: 07Mana
[7/300] Fetching Hiscores: 07Ashley
[8/300] Fetching Hiscores: 07wan
[9/300] Fetching Hiscores: 07Chris
[10/300] Fetching Hiscores: 07 0
[11/300] Fetching Hiscores: 07 Preacher
[12/300] Fetching Hiscores: 07 tim
[13/300] Fetching Hiscores: 07 PRISMA
[14/300] Fetching Hiscores: 07 t bone
[15/300] Fetching Hiscores: 07Geno
[16/300] Fetching Hiscores: 07TroyScape
[17/300] Fetching Hiscores: 07 maisy
[18/300] Fetching Hiscores: 07 ezra
[19/300] Fetching Hiscores: 07 Account
[20/300] Fetching Hiscores: 07 Camry
[21/300] Fetching Hiscores: Q7L
[22/300] Fetching Hiscores: q7f
Failed: q7f - HTTP 404
[23/300] Fetching Hiscores: q7d
[24/300] Fetching Hiscores: Q7 1
Failed: Q7 1 - HTTP 404
[25/300] Fetching Hiscores: q7g
[26/300] Fetching Hiscores: q7s
Failed: q7s - HTTP 404
[27/300]

[201/300] Fetching Hiscores: 4pf AddMan
[202/300] Fetching Hiscores: 4PF
[203/300] Fetching Hiscores: 4pf Leo
[204/300] Fetching Hiscores: 4pf Big T
[205/300] Fetching Hiscores: 4pf danaw1te
[206/300] Fetching Hiscores: 4pf Gooner
[207/300] Fetching Hiscores: 4pearlpawya
[208/300] Fetching Hiscores: 4peman
[209/300] Fetching Hiscores: 4pf cameron
Failed: 4pf cameron - HTTP 404
[210/300] Fetching Hiscores: 4pf Steve
[211/300] Fetching Hiscores: 4PLATE DEAD
[212/300] Fetching Hiscores: 4pf Eiji
[213/300] Fetching Hiscores: 4pf Danny
Failed: 4pf Danny - HTTP 404
[214/300] Fetching Hiscores: 4pf PvM Lao
[215/300] Fetching Hiscores: 4pf legit
Failed: 4pf legit - HTTP 404
[216/300] Fetching Hiscores: 4pf CaSh
Failed: 4pf CaSh - HTTP 404
[217/300] Fetching Hiscores: 4pf Dracula
Failed: 4pf Dracula - HTTP 404
[218/300] Fetching Hiscores: 4packofpbr
[219/300] Fetching Hiscores: 4pf tady
[220/300] Fetching Hiscores: 98qm8cm8q9mc
Failed: 98qm8cm8q9mc - HTTP 404
[221/300] Fetching Hiscores: 982
[2

,player,overall_rank,overall_level,overall_xp,attack_rank,attack_level,attack_xp,defence_rank,defence_level,defence_xp,...,runecraft_xp,hunter_rank,hunter_level,hunter_xp,construction_rank,construction_level,construction_xp,sailing_rank,sailing_level,sailing_xp
0,07 iron,382,2376,2186969492,1074,99,143756881,769,99,136216686,...,19572847,1960,99,70536444,443,99,34820520,12219,99,14164922
1,07dive,165,2376,3120241113,1678,99,110662107,949,99,115594812,...,46120817,1188,99,90675595,318,99,56103409,24224,99,13130933
2,07 Flare,66407,2341,593892412,40984,99,29757504,39589,99,24209970,...,14707818,55304,99,14340193,57144,99,13251248,299490,64,422540
3,07 Nihilist,106600,2278,646176615,20348,99,38454575,20340,99,29791019,...,13336934,115023,99,13087926,19946,99,13577899,-1,1,0
4,07 Hollyman,22611,2376,550419287,32283,99,32447653,36388,99,24831178,...,13116186,60220,99,14030170,98244,99,13131220,30350,99,13069755


## 13. Inspect collected Hiscores dataset

Before saving the dataset, basic checks are performed.

In [49]:
df_hiscores.shape

(248, 76)

In [50]:
df_hiscores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248 entries, 0 to 247
Data columns (total 76 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   player              248 non-null    object
 1   overall_rank        248 non-null    int64 
 2   overall_level       248 non-null    int64 
 3   overall_xp          248 non-null    int64 
 4   attack_rank         248 non-null    int64 
 5   attack_level        248 non-null    int64 
 6   attack_xp           248 non-null    int64 
 7   defence_rank        248 non-null    int64 
 8   defence_level       248 non-null    int64 
 9   defence_xp          248 non-null    int64 
 10  strength_rank       248 non-null    int64 
 11  strength_level      248 non-null    int64 
 12  strength_xp         248 non-null    int64 
 13  hitpoints_rank      248 non-null    int64 
 14  hitpoints_level     248 non-null    int64 
 15  hitpoints_xp        248 non-null    int64 
 16  ranged_rank         248 no

In [51]:
df_hiscores.head()

,player,overall_rank,overall_level,overall_xp,attack_rank,attack_level,attack_xp,defence_rank,defence_level,defence_xp,...,runecraft_xp,hunter_rank,hunter_level,hunter_xp,construction_rank,construction_level,construction_xp,sailing_rank,sailing_level,sailing_xp
0,07 iron,382,2376,2186969492,1074,99,143756881,769,99,136216686,...,19572847,1960,99,70536444,443,99,34820520,12219,99,14164922
1,07dive,165,2376,3120241113,1678,99,110662107,949,99,115594812,...,46120817,1188,99,90675595,318,99,56103409,24224,99,13130933
2,07 Flare,66407,2341,593892412,40984,99,29757504,39589,99,24209970,...,14707818,55304,99,14340193,57144,99,13251248,299490,64,422540
3,07 Nihilist,106600,2278,646176615,20348,99,38454575,20340,99,29791019,...,13336934,115023,99,13087926,19946,99,13577899,-1,1,0
4,07 Hollyman,22611,2376,550419287,32283,99,32447653,36388,99,24831178,...,13116186,60220,99,14030170,98244,99,13131220,30350,99,13069755


In [52]:
len(failed_players), failed_players[:20]

(52,
 ['q7f',
  'Q7 1',
  'q7s',
  'q7a tempest',
  'Q7ls Scout',
  'Kaeleen',
  'KalNun',
  'kamille goch',
  'Kappadalism',
  'bweakfest',
  'bwenno',
  'bwenton',
  'zrado',
  'UmphrayMcgee',
  'Umprella',
  'umpanii',
  'umpol',
  'ump45',
  'Qcid',
  'Qc99'])

## 14. Save the automatically collected Hiscores dataset

The final dataset is saved into the processed data folder.

This file will be used as input for the data cleaning notebook.

In [53]:
output_path = PROCESSED_DATA_DIR / "osrs_hiscores_auto_sample.csv"

df_hiscores.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

output_path

WindowsPath('C:/Projects/osrs-player-segmentation/data/processed/osrs_hiscores_auto_sample.csv')

## Summary

This notebook collected a larger and more diverse OSRS player sample.

The main steps were:

- generating random partial username search terms,
- collecting player names from public player search results,
- removing duplicate usernames,
- downloading public OSRS Hiscores data,
- parsing skill rank, level and experience values,
- saving the final dataset for cleaning and clustering.

The collected sample is not a perfectly random representation of all OSRS players, but it reduces the strong bias that would come from using only top leaderboard players.